## Microsoft Teams Video: SFM Calibration Pilot

This notebook applies the same calibration logic used in `Shibuya_WC2026.ipynb` and `Bottleneck_flow.ipynb` to the Teams video.

Because this video is a real-world camera view, not a controlled trajectory dataset, the workflow is:

```text
video pixels -> homography -> tracked pedestrian points -> metre trajectories -> SFM targets
```

The bottleneck notebook is still useful as the model-calibration reference: once this notebook extracts speed, spacing, and flow, those become the observed targets for SFM parameter experiments.

### 1. Setup

The source video has been copied into the project so all paths are relative and reproducible.

In [ ]:
from pathlib import Path
import math
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import cv2
    OPENCV_AVAILABLE = True
except ImportError:
    OPENCV_AVAILABLE = False
    print("OpenCV is not installed. Video extraction, homography, and optical-flow tracking will be skipped.")

try:
    display
except NameError:
    def display(obj):
        print(obj)


def find_project_dir():
    candidates = [
        Path.cwd(),
        Path.cwd() / "Documentation" / "Task 1 - NTU Paper + GABM",
        Path.cwd().parent,
    ]
    for candidate in candidates:
        video = candidate / "data" / "teams_video" / "source" / "MicrosoftTeams-video.mp4"
        if video.exists():
            return candidate.resolve()
    return Path.cwd().resolve()


PROJECT_DIR = find_project_dir()
DATA_DIR = PROJECT_DIR / "data" / "teams_video"
SOURCE_DIR = DATA_DIR / "source"
FRAME_DIR = DATA_DIR / "frames"
POINTS_DIR = DATA_DIR / "homography_points"
ANNOTATION_DIR = DATA_DIR / "annotations"
OUTPUT_DIR = DATA_DIR / "outputs"

for folder in [FRAME_DIR, POINTS_DIR, ANNOTATION_DIR, OUTPUT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

VIDEO_ID = "teams_video"
VIDEO_PATH = SOURCE_DIR / "MicrosoftTeams-video.mp4"
FPS_FALLBACK = 30.0

pd.set_option("display.max_columns", 80)
print("Project folder:", PROJECT_DIR)
print("Video path:", VIDEO_PATH)
print("OpenCV available:", OPENCV_AVAILABLE)

### 2. Inspect Video Metadata

The FPS and frame count determine the time step for speed calculations.

In [ ]:
def get_video_metadata(video_path):
    if not OPENCV_AVAILABLE:
        return {"path": str(video_path), "opened": False, "reason": "OpenCV unavailable"}

    cap = cv2.VideoCapture(str(video_path))
    opened = cap.isOpened()
    metadata = {"path": str(video_path), "opened": opened}
    if opened:
        fps = cap.get(cv2.CAP_PROP_FPS)
        frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
        width = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
        height = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
        metadata.update({
            "fps": fps,
            "frame_count": frame_count,
            "duration_s": frame_count / fps if fps else np.nan,
            "width_px": int(width),
            "height_px": int(height),
        })
    cap.release()
    return metadata


video_meta = get_video_metadata(VIDEO_PATH)
display(pd.DataFrame([video_meta]))
FPS = float(video_meta.get("fps") or FPS_FALLBACK)
print("FPS used for timing:", FPS)

### 3. Extract Reference Frames

Use a frame where ground-plane features are visible and pedestrian positions can be tracked. The 5s/10s frames are good starting points for this video.

In [ ]:
def extract_video_frame(video_path, time_s, out_dir=FRAME_DIR, prefix=VIDEO_ID):
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{prefix}_t{int(round(time_s)):03d}s.jpg"
    if out_path.exists():
        return out_path

    if not OPENCV_AVAILABLE:
        print("OpenCV unavailable; cannot extract frame:", out_path)
        return None

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        cap.release()
        raise ValueError(f"Could not open video: {video_path}")

    cap.set(cv2.CAP_PROP_POS_MSEC, time_s * 1000)
    ok, frame = cap.read()
    cap.release()
    if not ok:
        raise ValueError(f"Could not read frame at {time_s}s from {video_path}")

    cv2.imwrite(str(out_path), frame)
    return out_path


REFERENCE_TIMES_S = [0.0, 2.0, 5.0, 10.0, 15.0, 20.0, 30.0]
reference_frame_paths = [extract_video_frame(VIDEO_PATH, t) for t in REFERENCE_TIMES_S]
reference_frame_paths = [p for p in reference_frame_paths if p is not None]
REFERENCE_FRAME_PATH = FRAME_DIR / "teams_video_t005s.jpg"
print("Reference frame:", REFERENCE_FRAME_PATH)
reference_frame_paths

### 4. Show Reference Frame With Pixel Grid

Use this grid or `coords.py` to read pixel coordinates. `x_px` increases left-to-right, `y_px` increases top-to-bottom.

In [ ]:
def show_reference_with_pixel_grid(frame_path, grid_step=50, figsize=(7, 10)):
    if not OPENCV_AVAILABLE:
        print("OpenCV unavailable; cannot load image.")
        return
    if not Path(frame_path).exists():
        print("Frame does not exist yet:", frame_path)
        return

    image_bgr = cv2.imread(str(frame_path))
    if image_bgr is None:
        print("Could not load frame:", frame_path)
        return

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    h, w = image_rgb.shape[:2]

    plt.figure(figsize=figsize)
    plt.imshow(image_rgb)
    plt.xticks(np.arange(0, w + 1, grid_step), rotation=45)
    plt.yticks(np.arange(0, h + 1, grid_step))
    plt.grid(color="yellow", alpha=0.35)
    plt.xlim(0, w)
    plt.ylim(h, 0)
    plt.title(f"Pixel grid: {Path(frame_path).name}")
    plt.show()


show_reference_with_pixel_grid(REFERENCE_FRAME_PATH, grid_step=50)

### 5. Homography Point Table

Fill at least four road/plaza-plane points with both pixel coordinates and metre coordinates.

Important: this video has perspective, so do **not** use a single pixel-to-metre ratio. Homography is the scale transform.

Use only points on the same flat walking surface. Do not use roof beams, building corners, signs, raised objects, or people.

In [ ]:
POINT_COLUMNS = ["point_id", "x_px", "y_px", "x_m", "y_m", "notes"]
POINT_TABLE_PATH = POINTS_DIR / f"{VIDEO_ID}_homography_points.csv"

point_table = pd.DataFrame([
    ["P1", np.nan, np.nan, np.nan, np.nan, "ground-plane point; fill pixel and metre coordinates"],
    ["P2", np.nan, np.nan, np.nan, np.nan, "ground-plane point; fill pixel and metre coordinates"],
    ["P3", np.nan, np.nan, np.nan, np.nan, "ground-plane point; fill pixel and metre coordinates"],
    ["P4", np.nan, np.nan, np.nan, np.nan, "ground-plane point; fill pixel and metre coordinates"],
    ["P5", np.nan, np.nan, np.nan, np.nan, "optional extra point for a more stable fit"],
    ["P6", np.nan, np.nan, np.nan, np.nan, "optional extra point for a more stable fit"],
], columns=POINT_COLUMNS)

if POINT_TABLE_PATH.exists():
    point_table = pd.read_csv(POINT_TABLE_PATH)
    for col in POINT_COLUMNS:
        if col not in point_table.columns:
            point_table[col] = np.nan
    point_table = point_table[POINT_COLUMNS]
else:
    point_table.to_csv(POINT_TABLE_PATH, index=False)
    print("Created blank point table:", POINT_TABLE_PATH)

display(point_table)

### 5.1 Coordinate Picker Command

Run this in a terminal from the notebook folder to click homography points.

After clicking, fill `x_m` and `y_m` manually using measured real-world distances.

In [ ]:
def relative_or_absolute(path, base=PROJECT_DIR):
    path = Path(path).resolve()
    try:
        return str(path.relative_to(base.resolve()))
    except ValueError:
        return str(path)


coords_command = (
    f'python coords.py "{relative_or_absolute(REFERENCE_FRAME_PATH)}" '
    f'--out "{relative_or_absolute(POINT_TABLE_PATH)}" --n 6'
)
print("Run this from the project/notebook folder:")
print(coords_command)

### 6. Compute Homography

This converts pixel coordinates to real-world metres.

In [ ]:
def validate_point_table(points_df):
    required = ["x_px", "y_px", "x_m", "y_m"]
    missing = [col for col in required if col not in points_df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    clean = points_df.copy()
    for col in required:
        clean[col] = pd.to_numeric(clean[col], errors="coerce")
    clean = clean.dropna(subset=required)
    if len(clean) < 4:
        return clean, f"Need at least 4 complete point pairs; currently have {len(clean)}."
    return clean, None


def compute_homography(points_df):
    clean, message = validate_point_table(points_df)
    if message is not None:
        return None, clean, message
    if not OPENCV_AVAILABLE:
        return None, clean, "OpenCV unavailable."

    image_points = clean[["x_px", "y_px"]].to_numpy(dtype=np.float32)
    world_points = clean[["x_m", "y_m"]].to_numpy(dtype=np.float32)
    H, status = cv2.findHomography(image_points, world_points, method=0)
    if H is None:
        return None, clean, "Homography failed. Recheck point ordering and coordinates."
    return H, clean, None


H_img_to_world, homography_points, homography_message = compute_homography(point_table)
if homography_message:
    print(homography_message)
else:
    print("Homography image -> world matrix:")
    print(H_img_to_world)
    display(homography_points)

### 7. Test Homography Mapping

Residual error checks whether the clicked pixel points map back to the manually entered metre coordinates.

In [ ]:
def pixels_to_world(points_px, H):
    if H is None or not OPENCV_AVAILABLE:
        return np.full((len(points_px), 2), np.nan)
    points_px = np.asarray(points_px, dtype=np.float32).reshape(-1, 1, 2)
    points_m = cv2.perspectiveTransform(points_px, H)
    return points_m.reshape(-1, 2)


def homography_residual_report(points_df, H):
    if H is None or points_df.empty:
        return pd.DataFrame()
    mapped = pixels_to_world(points_df[["x_px", "y_px"]].to_numpy(), H)
    report = points_df.copy()
    report["pred_x_m"] = mapped[:, 0]
    report["pred_y_m"] = mapped[:, 1]
    report["error_m"] = np.sqrt((report["pred_x_m"] - report["x_m"]) ** 2 + (report["pred_y_m"] - report["y_m"]) ** 2)
    return report


residual_report = homography_residual_report(homography_points, H_img_to_world)
if residual_report.empty:
    print("No homography residuals yet. Fill the point table first.")
else:
    display(residual_report)
    print("Mean residual error (m):", residual_report["error_m"].mean())

### 8. Click Initial Pedestrian Points

Use `coords.py` to select pedestrian foot/ground-contact points on the starting frame. This creates the starting points for optical-flow tracking.

Start small: 10 to 20 pedestrians is enough for a pilot.

In [ ]:
PEDESTRIAN_START_TIME_S = 5.0
PEDESTRIAN_START_FRAME_PATH = FRAME_DIR / f"{VIDEO_ID}_t{int(PEDESTRIAN_START_TIME_S):03d}s.jpg"
PEDESTRIAN_START_CSV = ANNOTATION_DIR / f"{VIDEO_ID}_pedestrians_t{int(PEDESTRIAN_START_TIME_S):03d}s_points.csv"

ped_command = (
    f'python coords.py "{relative_or_absolute(PEDESTRIAN_START_FRAME_PATH)}" '
    f'--out "{relative_or_absolute(PEDESTRIAN_START_CSV)}" --n 10'
)
print("Run this from the project/notebook folder to click pedestrian foot points:")
print(ped_command)

### 9. Auto-Track Selected Pedestrians

This follows the initially clicked pedestrian points through the video using sparse Lucas-Kanade optical flow.

Inspect the output before treating it as calibration data. Optical flow can drift when pedestrians occlude each other or the point is not on a strong visual feature.

In [ ]:
TRACK_COLUMNS = ["video_id", "ped_id", "frame", "time_s", "x_px", "y_px", "visibility", "notes"]
TRACK_START_CSV = PEDESTRIAN_START_CSV
TRACK_START_TIME_S = PEDESTRIAN_START_TIME_S
TRACK_END_TIME_S = 20.0
TRACK_SAMPLE_STEP_S = 0.5
WRITE_AUTO_TRACKS_TO_MANUAL_CSV = False
TRACK_PATH = ANNOTATION_DIR / f"{VIDEO_ID}_manual_tracks.csv"


def track_clicked_points_lk(video_path, start_csv, start_time_s, end_time_s, sample_step_s=0.5, fps=FPS, video_id=VIDEO_ID):
    if not OPENCV_AVAILABLE:
        print("OpenCV unavailable; cannot track points.")
        return pd.DataFrame()
    start_csv = Path(start_csv)
    if not start_csv.exists():
        print("Starting point CSV not found:", start_csv)
        return pd.DataFrame()

    starts = pd.read_csv(start_csv)
    starts["x_px"] = pd.to_numeric(starts["x_px"], errors="coerce")
    starts["y_px"] = pd.to_numeric(starts["y_px"], errors="coerce")
    starts = starts.dropna(subset=["x_px", "y_px"]).reset_index(drop=True)
    if starts.empty:
        print("No starting points found in", start_csv)
        return pd.DataFrame()

    ped_ids = [f"P{i:02d}" for i in range(1, len(starts) + 1)]
    point_labels = starts["point_id"].astype(str).to_list() if "point_id" in starts.columns else ped_ids
    p0 = starts[["x_px", "y_px"]].to_numpy(dtype=np.float32).reshape(-1, 1, 2)

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise ValueError(f"Could not open video: {video_path}")

    start_frame = int(round(start_time_s * fps))
    end_frame = int(round(end_time_s * fps))
    sample_step_frames = max(1, int(round(sample_step_s * fps)))
    sample_frames = set(range(start_frame, end_frame + 1, sample_step_frames))

    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    ok, first_frame = cap.read()
    if not ok:
        cap.release()
        raise ValueError(f"Could not read start frame {start_frame}")

    prev_gray = cv2.cvtColor(first_frame, cv2.COLOR_BGR2GRAY)
    prev_pts = p0.copy()
    active = np.ones(len(ped_ids), dtype=bool)
    rows = []
    lk_params = dict(
        winSize=(21, 21),
        maxLevel=3,
        criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01),
        minEigThreshold=1e-4,
    )

    def append_rows(frame_idx, pts, status_text):
        time_s = frame_idx / fps
        for idx, (ped_id, label) in enumerate(zip(ped_ids, point_labels)):
            if active[idx] and np.isfinite(pts[idx, 0, 0]) and np.isfinite(pts[idx, 0, 1]):
                x_px = float(pts[idx, 0, 0])
                y_px = float(pts[idx, 0, 1])
                visibility = "tracked"
            else:
                x_px = np.nan
                y_px = np.nan
                visibility = "lost"
            rows.append({
                "video_id": video_id,
                "ped_id": ped_id,
                "frame": frame_idx,
                "time_s": time_s,
                "x_px": x_px,
                "y_px": y_px,
                "visibility": visibility,
                "notes": f"{label}; LK optical flow; {status_text}",
            })

    append_rows(start_frame, prev_pts, "initial coords.py click")
    for frame_idx in range(start_frame + 1, end_frame + 1):
        ok, frame = cap.read()
        if not ok:
            print("Video ended before requested end frame:", frame_idx)
            break
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        next_pts, status, err = cv2.calcOpticalFlowPyrLK(prev_gray, gray, prev_pts, None, **lk_params)
        if next_pts is None or status is None:
            active[:] = False
            next_pts = prev_pts.copy()
        else:
            status = status.reshape(-1).astype(bool)
            active &= status
            next_pts[~active] = np.nan
        if frame_idx in sample_frames:
            append_rows(frame_idx, next_pts, "sampled tracked point")
        prev_gray = gray
        prev_pts = next_pts
    cap.release()
    return pd.DataFrame(rows, columns=TRACK_COLUMNS)


def map_track_rows_to_world(track_rows, H):
    out = track_rows.copy()
    out["x_m"] = np.nan
    out["y_m"] = np.nan
    if out.empty:
        return out
    if H is None:
        print("Homography is not ready, so world coordinates remain blank.")
        return out
    for col in ["frame", "time_s", "x_px", "y_px"]:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    valid = out[["x_px", "y_px"]].notna().all(axis=1)
    if valid.any():
        mapped = pixels_to_world(out.loc[valid, ["x_px", "y_px"]].to_numpy(), H)
        out.loc[valid, "x_m"] = mapped[:, 0]
        out.loc[valid, "y_m"] = mapped[:, 1]
    return out


auto_track_rows_px = track_clicked_points_lk(VIDEO_PATH, TRACK_START_CSV, TRACK_START_TIME_S, TRACK_END_TIME_S, TRACK_SAMPLE_STEP_S)
if auto_track_rows_px.empty:
    print("No auto-tracks produced yet.")
else:
    auto_track_path = ANNOTATION_DIR / f"{VIDEO_ID}_lk_tracks_px.csv"
    auto_track_rows_px.to_csv(auto_track_path, index=False)
    print("Saved pixel tracks:", auto_track_path)
    display(auto_track_rows_px.head(20))

    auto_tracks_world = map_track_rows_to_world(auto_track_rows_px, H_img_to_world)
    auto_world_path = ANNOTATION_DIR / f"{VIDEO_ID}_lk_tracks_world.csv"
    auto_tracks_world.to_csv(auto_world_path, index=False)
    print("Saved world-coordinate tracks:", auto_world_path)
    display(auto_tracks_world.head(20))

    status_summary = auto_track_rows_px.groupby(["ped_id", "visibility"], as_index=False).size()
    display(status_summary)

    if WRITE_AUTO_TRACKS_TO_MANUAL_CSV:
        auto_track_rows_px.to_csv(TRACK_PATH, index=False)
        print("Wrote auto-tracks to manual-track CSV:", TRACK_PATH)
    else:
        print("Review tracks first. Set WRITE_AUTO_TRACKS_TO_MANUAL_CSV = True to use them as the main track CSV.")

### 10. Compute Speed And Direction

These observed trajectories become SFM calibration targets.

In [ ]:
def compute_track_kinematics(tracks_world, min_dt=1e-6):
    if tracks_world.empty:
        return pd.DataFrame()
    required = ["video_id", "ped_id", "time_s", "x_m", "y_m"]
    missing = [col for col in required if col not in tracks_world.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    out = tracks_world.dropna(subset=["ped_id", "time_s", "x_m", "y_m"]).copy()
    if out.empty:
        return out
    out = out.sort_values(["video_id", "ped_id", "time_s"])
    group_cols = ["video_id", "ped_id"]
    out["prev_x_m"] = out.groupby(group_cols)["x_m"].shift(1)
    out["prev_y_m"] = out.groupby(group_cols)["y_m"].shift(1)
    out["prev_time_s"] = out.groupby(group_cols)["time_s"].shift(1)
    out["dx_m"] = out["x_m"] - out["prev_x_m"]
    out["dy_m"] = out["y_m"] - out["prev_y_m"]
    out["dt_s"] = out["time_s"] - out["prev_time_s"]
    out.loc[out["dt_s"] <= min_dt, "dt_s"] = np.nan
    out["step_distance_m"] = np.sqrt(out["dx_m"] ** 2 + out["dy_m"] ** 2)
    out["speed_mps"] = out["step_distance_m"] / out["dt_s"]
    out["direction_deg"] = np.degrees(np.arctan2(out["dy_m"], out["dx_m"]))
    return out


tracks_world_path = ANNOTATION_DIR / f"{VIDEO_ID}_lk_tracks_world.csv"
tracks_world = pd.read_csv(tracks_world_path) if tracks_world_path.exists() else pd.DataFrame()
kinematics = compute_track_kinematics(tracks_world)
if kinematics.empty:
    print("No kinematics yet. Complete homography and tracking first.")
else:
    display(kinematics.head())

### 11. Summarise Speed Targets

Running speeds are allowed. Very high speeds are flagged for review rather than automatically discarded.

In [ ]:
REVIEW_SPEED_ABOVE_MPS = 7.0


def summarise_speeds(kinematics, review_speed_above=REVIEW_SPEED_ABOVE_MPS):
    if kinematics.empty or "speed_mps" not in kinematics.columns:
        return pd.DataFrame()
    clean = kinematics.replace([np.inf, -np.inf], np.nan).dropna(subset=["speed_mps"])
    clean = clean[clean["speed_mps"] >= 0]
    if clean.empty:
        return pd.DataFrame()

    high_speed = clean[clean["speed_mps"] > review_speed_above]
    if not high_speed.empty:
        print(f"Review note: {len(high_speed)} observations exceed {review_speed_above} m/s. They are retained.")
        display(high_speed[["video_id", "ped_id", "time_s", "x_m", "y_m", "speed_mps", "direction_deg"]].head(20))

    return pd.DataFrame([{
        "video_id": VIDEO_ID,
        "n_speed_observations": len(clean),
        "n_pedestrians": clean["ped_id"].nunique(),
        "mean_speed_mps": clean["speed_mps"].mean(),
        "median_speed_mps": clean["speed_mps"].median(),
        "std_speed_mps": clean["speed_mps"].std(),
        "p10_speed_mps": clean["speed_mps"].quantile(0.10),
        "p90_speed_mps": clean["speed_mps"].quantile(0.90),
        "p95_speed_mps": clean["speed_mps"].quantile(0.95),
        "max_speed_mps": clean["speed_mps"].max(),
        "n_above_review_speed": len(high_speed),
        "review_speed_above_mps": review_speed_above,
        "mean_direction_deg": clean["direction_deg"].mean(),
    }])


speed_summary = summarise_speeds(kinematics)
if speed_summary.empty:
    print("No speed summary yet.")
else:
    display(speed_summary)

### 12. Estimate Nearest-Neighbour Spacing

Spacing helps tune social-force repulsion and personal-space assumptions.

In [ ]:
def nearest_neighbour_spacing(tracks_world, time_round_s=0.5):
    if tracks_world.empty:
        return pd.DataFrame()
    df = tracks_world.dropna(subset=["ped_id", "time_s", "x_m", "y_m"]).copy()
    if df.empty:
        return pd.DataFrame()
    df["time_bin_s"] = (df["time_s"] / time_round_s).round() * time_round_s
    rows = []
    for (video_id, time_bin), group in df.groupby(["video_id", "time_bin_s"]):
        if len(group) < 2:
            continue
        coords = group[["x_m", "y_m"]].to_numpy(dtype=float)
        ped_ids = group["ped_id"].to_list()
        diff = coords[:, None, :] - coords[None, :, :]
        dist = np.sqrt((diff ** 2).sum(axis=2))
        np.fill_diagonal(dist, np.nan)
        nn = np.nanmin(dist, axis=1)
        for ped_id, nn_m in zip(ped_ids, nn):
            rows.append({"video_id": video_id, "time_bin_s": time_bin, "ped_id": ped_id, "nn_spacing_m": nn_m})
    return pd.DataFrame(rows)


spacing = nearest_neighbour_spacing(tracks_world)
if spacing.empty:
    print("No spacing estimates yet.")
else:
    spacing_summary = spacing.groupby("video_id", as_index=False).agg(
        mean_nn_spacing_m=("nn_spacing_m", "mean"),
        median_nn_spacing_m=("nn_spacing_m", "median"),
        p10_nn_spacing_m=("nn_spacing_m", lambda s: s.quantile(0.10)),
    )
    display(spacing_summary)

### 13. Estimate Flow Across A Measurement Line

Set a line in world coordinates after homography is ready. This validates throughput, similar to `Bottleneck_flow.ipynb`.

In [ ]:
def compute_line_flow(kinematics, axis="y", line_value=0.0, bin_s=1.0):
    if kinematics.empty:
        return pd.DataFrame()
    pos_col = f"{axis}_m"
    prev_col = f"prev_{axis}_m"
    if pos_col not in kinematics.columns or prev_col not in kinematics.columns:
        raise ValueError(f"Expected columns {pos_col} and {prev_col}")
    df = kinematics.dropna(subset=[pos_col, prev_col, "time_s", "ped_id"]).copy()
    if df.empty:
        return pd.DataFrame()
    crossed = ((df[prev_col] - line_value) * (df[pos_col] - line_value)) <= 0
    crossings = df[crossed].copy()
    if crossings.empty:
        return pd.DataFrame()
    crossings["time_bin_s"] = np.floor(crossings["time_s"] / bin_s) * bin_s
    flow = crossings.groupby(["video_id", "time_bin_s"], as_index=False)["ped_id"].nunique()
    flow = flow.rename(columns={"ped_id": "crossings"})
    flow["flow_p_per_s"] = flow["crossings"] / bin_s
    return flow


FLOW_AXIS = "y"
FLOW_LINE_VALUE_M = 0.0
flow = compute_line_flow(kinematics, axis=FLOW_AXIS, line_value=FLOW_LINE_VALUE_M, bin_s=1.0)
if flow.empty:
    print("No flow estimate yet. Define a meaningful world-coordinate measurement line first.")
else:
    display(flow.head())

### 14. Save Calibration Targets

This table is the observed target set for SFM calibration: speed, spacing, and flow.

In [ ]:
def combine_calibration_targets(speed_summary, spacing=None, flow=None):
    if speed_summary is None or speed_summary.empty:
        return pd.DataFrame()
    out = speed_summary.copy()
    if spacing is not None and not spacing.empty:
        spacing_summary = spacing.groupby("video_id", as_index=False).agg(
            mean_nn_spacing_m=("nn_spacing_m", "mean"),
            median_nn_spacing_m=("nn_spacing_m", "median"),
        )
        out = out.merge(spacing_summary, on="video_id", how="left")
    if flow is not None and not flow.empty:
        flow_summary = flow.groupby("video_id", as_index=False).agg(
            mean_flow_p_per_s=("flow_p_per_s", "mean"),
            max_flow_p_per_s=("flow_p_per_s", "max"),
        )
        out = out.merge(flow_summary, on="video_id", how="left")
    return out


calibration_targets = combine_calibration_targets(speed_summary, spacing, flow)
if calibration_targets.empty:
    print("No calibration target CSV saved yet because calibration/tracking values are incomplete.")
else:
    out_path = OUTPUT_DIR / f"{VIDEO_ID}_manual_calibration_targets.csv"
    calibration_targets.to_csv(out_path, index=False)
    print("Saved:", out_path)
    display(calibration_targets)

### 15. SFM Parameter Calibration Scaffold

Use the extracted targets like the bottleneck notebook. Start with a small parameter set before running a full Genetic Algorithm.

In [ ]:
sfm_parameter_plan = pd.DataFrame([
    {"config_section": "desired_force", "parameter": "relaxation_time", "current_value": 0.50, "candidate_low": 0.30, "candidate_high": 1.00, "target_metrics": "speed smoothness, mean_speed_mps"},
    {"config_section": "desired_force", "parameter": "factor", "current_value": 1.00, "candidate_low": 0.50, "candidate_high": 2.00, "target_metrics": "mean_speed_mps, flow_p_per_s"},
    {"config_section": "scene", "parameter": "max_speed_multiplier", "current_value": 1.30, "candidate_low": 1.10, "candidate_high": 1.80, "target_metrics": "p90/p95/max speed"},
    {"config_section": "social_force", "parameter": "factor", "current_value": 5.10, "candidate_low": 1.00, "candidate_high": 10.00, "target_metrics": "spacing and flow"},
    {"config_section": "social_force", "parameter": "lambda_importance", "current_value": 2.00, "candidate_low": 0.50, "candidate_high": 4.00, "target_metrics": "directional avoidance"},
    {"config_section": "social_force", "parameter": "gamma", "current_value": 0.35, "candidate_low": 0.10, "candidate_high": 0.80, "target_metrics": "spacing and steering response"},
])
display(sfm_parameter_plan)


def normalised_metric_mse(simulated, observed, metric_cols, key_col="video_id", weights=None):
    weights = weights or {}
    if simulated is None or observed is None or simulated.empty or observed.empty:
        return np.inf, pd.DataFrame()
    merged = observed.merge(simulated, on=key_col, suffixes=("_obs", "_sim"))
    if merged.empty:
        return np.inf, pd.DataFrame()
    rows = []
    total = 0.0
    total_weight = 0.0
    for metric in metric_cols:
        obs_col = f"{metric}_obs"
        sim_col = f"{metric}_sim"
        if obs_col not in merged.columns or sim_col not in merged.columns:
            continue
        obs = pd.to_numeric(merged[obs_col], errors="coerce")
        sim = pd.to_numeric(merged[sim_col], errors="coerce")
        valid = obs.notna() & sim.notna()
        if not valid.any():
            continue
        scale = max(float(obs[valid].abs().median()), 1e-6)
        metric_mse = float((((sim[valid] - obs[valid]) / scale) ** 2).mean())
        weight = float(weights.get(metric, 1.0))
        rows.append({"metric": metric, "normalised_mse": metric_mse, "weight": weight})
        total += weight * metric_mse
        total_weight += weight
    return (total / total_weight if total_weight else np.inf), pd.DataFrame(rows)


metric_cols = ["mean_speed_mps", "median_speed_mps", "std_speed_mps", "mean_nn_spacing_m", "mean_flow_p_per_s"]
metric_weights = {"mean_speed_mps": 1.0, "median_speed_mps": 1.0, "std_speed_mps": 0.5, "mean_nn_spacing_m": 1.5, "mean_flow_p_per_s": 2.0}
print("Objective ready. Next required function: simulate_teams_sfm_with_params(params).")

### 16. Parameter Sampling Skeleton

This creates candidate parameter sets. Replace random sampling with a Genetic Algorithm after the simulation bridge exists.

In [ ]:
def sample_parameter_sets(parameter_plan, n=20, seed=42):
    rng = np.random.default_rng(seed)
    rows = []
    for sample_id in range(n):
        row = {"sample_id": sample_id}
        for _, param in parameter_plan.iterrows():
            name = f"{param['config_section']}.{param['parameter']}"
            row[name] = rng.uniform(param["candidate_low"], param["candidate_high"])
        rows.append(row)
    return pd.DataFrame(rows)


def simulate_teams_sfm_with_params(params):
    """TODO: connect this to a PySocialForce scenario matching the Teams video.

    Expected return: one row with video_id and simulated metric columns matching calibration_targets.
    """
    return pd.DataFrame()


parameter_samples = sample_parameter_sets(sfm_parameter_plan, n=10, seed=13)
display(parameter_samples)

### 17. Experiment Checklist

1. Choose a reference frame.
2. Select 4+ ground-plane homography points with `coords.py`.
3. Fill their real-world `x_m`, `y_m` values from measured plaza dimensions or a map/aerial reference.
4. Rerun homography and residual checks.
5. Click initial pedestrian foot points.
6. Run optical-flow tracking and inspect the saved tracks.
7. Compute speed, spacing, and flow.
8. Save calibration targets.
9. Use the SFM objective scaffold to tune desired/social-force parameters.

This mirrors Shibuya for trajectory extraction and Bottleneck for SFM target calibration.